# MIMICSplus — Mortality file preprocessing

Reads CLM h1 files (preprocessed by `preprocess_clm_input.bash`) and writes
per-year mortality `.nc` files in the format expected by MIMICSplus.

**Edit the CONFIGURATION cell before running.**

In [27]:
import os
import glob
import xarray as xr
import numpy as np

In [45]:
test

<xarray.Dataset> Size: 113kB
Dimensions:                        (time: 12, bnds: 2, lndgrid: 1, levsoi: 20,
                                    levdcmp: 25, cft: 2, glc_nec: 10, ltype: 9,
                                    natpft: 15, levlak: 10, nvegwcs: 4,
                                    hist_interval: 2, levgrnd: 25)
Coordinates:
  * time                           (time) object 96B 1852-02-01 00:00:00 ... ...
  * levsoi                         (levsoi) float32 80B 0.01 0.04 ... 6.94 8.03
  * levdcmp                        (levdcmp) float32 100B 0.01 0.04 ... 42.0
  * levlak                         (levlak) float32 40B 0.05 0.6 ... 34.33 44.78
  * levgrnd                        (levgrnd) float32 100B 0.01 0.04 ... 42.0
Dimensions without coordinates: bnds, lndgrid, cft, glc_nec, ltype, natpft,
                                nvegwcs, hist_interval
Data variables: (12/560)
    time_bnds                      (time, bnds) float64 192B ...
    mcdate                         (time) int32 48B ...
    mcsec                          (time) int32 48B ...
    mdcur                          (time) int32 48B ...
    mscur                          (time) int32 48B ...
    nstep                          (time) int32 48B ...
    ...                             ...
    FROOT_PROF                     (time, levdcmp, lndgrid) float32 1kB ...
    LEAF_PROF                      (time, levdcmp, lndgrid) float32 1kB ...
    NDEP_PROF                      (time, levdcmp, lndgrid) float32 1kB ...
    date_written                   (time) |S16 192B ...
    time_bounds                    (time, hist_interval) object 192B ...
    time_written                   (time) |S16 192B ...
Attributes: (12/43)
    CDI:                                  Climate Data Interface version 1.9....
    Conventions:                          CF-1.0
    history:                              Sun Jan  9 16:23:23 2022: ncks -A /...
    source:                               Community Terrestrial Systems Model
    title:                                CLM History file information
    comment:                              NOTE: None of the variables are wei...
    ...                                   ...
    time_period_freq:                     month_1
    Time_constant_3Dvars_filename:        ./31464_Hurdal_hist_for_decomp.clm2...
    Time_constant_3Dvars:                 ZSOI:DZSOI:WATSAT:SUCSAT:BSW:HKSAT:...
    CDO:                                  Climate Data Operators version 1.9....
    history_of_appended_files:            Sun Jan  9 16:23:23 2022: Appended ...
    NCO:                                  4.6.9

In [46]:
era

<xarray.Dataset> Size: 24kB
Dimensions:                         (time: 12, nbnd: 2, lndgrid: 1,
                                     levdcmp: 25, levgrnd: 25, levsoi: 20,
                                     levlak: 10)
Coordinates:
  * levgrnd                         (levgrnd) float32 100B 0.01 0.04 ... 42.0
  * levsoi                          (levsoi) float32 80B 0.01 0.04 ... 6.94 8.03
  * levlak                          (levlak) float32 40B 0.05 0.6 ... 44.78
  * levdcmp                         (levdcmp) float32 100B 0.01 0.04 ... 42.0
  * time                            (time) object 96B 1850-01-16 12:00:00 ......
Dimensions without coordinates: nbnd, lndgrid
Data variables: (12/96)
    mcdate                          (time) int32 48B ...
    mcsec                           (time) int32 48B ...
    mdcur                           (time) int32 48B ...
    mscur                           (time) int32 48B ...
    nstep                           (time) int32 48B ...
    time_bounds                     (time, nbnd) object 192B ...
    ...                              ...
    watsat                          (time, levgrnd, lndgrid) float32 1kB ...
    H2OSOI                          (time, levsoi, lndgrid) float32 960B ...
    SOILICE                         (time, levsoi, lndgrid) float32 960B ...
    SOILLIQ                         (time, levsoi, lndgrid) float32 960B ...
    T_SCALAR                        (time, levsoi, lndgrid) float32 960B ...
    W_SCALAR                        (time, levsoi, lndgrid) float32 960B ...
Attributes: (12/40)
    title:                                CLM History file information
    comment:                              NOTE: None of the variables are wei...
    Conventions:                          CF-1.0
    history:                              created on 04/20/26 16:31:33
    source:                               Community Terrestrial Systems Model
    hostname:                             betzy
    ...                                   ...
    ctype_urban_pervious_road:            75
    cft_c3_crop:                          1
    cft_c3_irrigated:                     2
    time_period_freq:                     month_1
    Time_constant_3Dvars_filename:        ./ERA5L_HIST1_Bygland_century.clm2....
    Time_constant_3Dvars:                 ZSOI:DZSOI:WATSAT:SUCSAT:BSW:HKSAT:...

## Configuration

In [19]:
# ── Site settings ──────────────────────────────────────────────────────────
SITE = "Bygland"

# Directory containing preprocessed h1 files (output of preprocess_clm_input.bash)
# Files are expected to match the pattern: *.clm2.h1.{YEAR}.nc
H1_DIR = f"/home/elisacw/mimicsplus_input/{SITE}/clm_h1"

# Where to write per-year mortality files
MORT_DIR = f"/home/elisacw/mimicsplus_input/{SITE}/mortality"

# Year range to process (inclusive)
YEAR_START = 1850
YEAR_END   = 2025

# The first year of the historical run — used to extract profiles (CROOT_PROF, STEM_PROF)
# which are constant but only need to be written once.
PROFILE_YEAR = 1850

# Spinup years (used to build the spinup mortality file)
SPINUP_START = 1850
SPINUP_END   = 1869

os.makedirs(MORT_DIR, exist_ok=True)
print(f"Site:       {SITE}")
print(f"Input dir:  {H1_DIR}")
print(f"Output dir: {MORT_DIR}")

Site:       Bygland
Input dir:  /home/elisacw/mimicsplus_input/Bygland/clm_h1
Output dir: /home/elisacw/mimicsplus_input/Bygland/mortality


## Variable definitions

In [20]:
# Variables to split between metabolic and structural litter
SPLIT_C = ["M_LEAFC_TO_LITTER",  "M_FROOTC_TO_LITTER"]
SPLIT_N = ["M_LEAFN_TO_LITTER",  "M_FROOTN_TO_LITTER"]

# Variables that go to metabolic litter only, grouped by the root profile they follow.
# Each key becomes a field in the output dataset.
MET_C = {
    "met_leaf_prof_mortC":  ["M_LEAFC_STORAGE_TO_LITTER",
                              "M_LEAFC_XFER_TO_LITTER",
                              "M_GRESP_STORAGE_TO_LITTER",
                              "M_GRESP_XFER_TO_LITTER"],
    "met_froot_prof_mortC": ["M_FROOTC_STORAGE_TO_LITTER",
                              "M_FROOTC_XFER_TO_LITTER"],
    "met_croot_prof_mortC": ["M_LIVECROOTC_XFER_TO_LITTER",
                              "M_DEADCROOTC_XFER_TO_LITTER",
                              "M_LIVECROOTC_STORAGE_TO_LITTER",
                              "M_DEADCROOTC_STORAGE_TO_LITTER"],
    "met_stem_prof_mortC":  ["M_LIVESTEMC_STORAGE_TO_LITTER",
                              "M_LIVESTEMC_XFER_TO_LITTER",
                              "M_DEADSTEMC_STORAGE_TO_LITTER",
                              "M_DEADSTEMC_XFER_TO_LITTER"],
}

MET_N = {
    "met_leaf_prof_mortN":  ["M_LEAFN_STORAGE_TO_LITTER",
                              "M_LEAFN_XFER_TO_LITTER",
                              "M_RETRANSN_TO_LITTER"],
    "met_froot_prof_mortN": ["M_FROOTN_STORAGE_TO_LITTER",
                              "M_FROOTN_XFER_TO_LITTER"],
    "met_croot_prof_mortN": ["M_LIVECROOTN_STORAGE_TO_LITTER",
                             
                              "M_LIVECROOTN_XFER_TO_LITTER",
                              "M_DEADCROOTN_XFER_TO_LITTER"],
    "met_stem_prof_mortN":  ["M_LIVESTEMN_STORAGE_TO_LITTER",
                              "M_DEADSTEMN_STORAGE_TO_LITTER",
                              "M_LIVESTEMN_XFER_TO_LITTER",
                              "M_DEADSTEMN_XFER_TO_LITTER"],
}

# "M_DEADCROOTN_STORAGE_TO_LITTER",

## Core processing function

In [21]:
def process_mortality(ds, include_profiles=False):
    """
    Extract and aggregate mortality variables from a CLM h1 xarray Dataset.

    Parameters
    ----------
    ds : xr.Dataset
        Opened CLM h1 dataset for one year.
    include_profiles : bool
        If True, also extract CROOT_PROF and STEM_PROF (needed for the first
        year of each simulation segment).

    Returns
    -------
    xr.Dataset
    """
    out = xr.Dataset()
    out["mcdate"] = ds["mcdate"]

    if include_profiles:
        out["CROOT_PROF"] = ds["CROOT_PROF"]
        out["STEM_PROF"]  = ds["STEM_PROF"]

    # Variables split between metabolic and structural
    out["split_leaf_prof_mortC"]  = ds["M_LEAFC_TO_LITTER"]
    out["split_froot_prof_mortC"] = ds["M_FROOTC_TO_LITTER"]
    out["split_leaf_prof_mortN"]  = ds["M_LEAFN_TO_LITTER"]
    out["split_froot_prof_mortN"] = ds["M_FROOTN_TO_LITTER"]

    # Summed metabolic C fluxes
    for out_var, src_vars in MET_C.items():
        out[out_var] = sum(ds[v] for v in src_vars)

    # Summed metabolic N fluxes
    for out_var, src_vars in MET_N.items():
        out[out_var] = sum(ds[v] for v in src_vars)

    return out

## Helper: find h1 file for a given year

In [22]:
def find_h1_file(year):
    pattern = os.path.join(H1_DIR, f"{SITE}_hist_all.{year}.nc")
    if not os.path.exists(pattern):
        raise FileNotFoundError(
            f"No h1 file for year {year}\n  Tried: {pattern}"
        )
    return pattern

## Process all years

In [23]:
skipped  = []
missing  = []
processed = []

for year in range(YEAR_START, YEAR_END + 1):
    out_path = os.path.join(MORT_DIR, f"mort_{SITE}_{year}.nc")

    if os.path.exists(out_path):
        skipped.append(year)
        continue

    try:
        h1_file = find_h1_file(year)
    except FileNotFoundError:
        missing.append(year)
        continue

    with xr.open_dataset(h1_file) as ds:
        include_profiles = (year == PROFILE_YEAR)
        ds_out = process_mortality(ds, include_profiles=include_profiles)

    ds_out.to_netcdf(out_path)
    processed.append(year)

print(f"Processed : {len(processed)} years")
print(f"Skipped   : {len(skipped)} years (already existed)")
if missing:
    print(f"Missing   : {len(missing)} years — {missing[:10]}{'...' if len(missing)>10 else ''}")

Processed : 175 years
Skipped   : 0 years (already existed)
Missing   : 1 years — [2025]


## Create spinup mortality file

In [25]:
spinup_out = os.path.join(MORT_DIR, f"mort_{SITE}_for_spinup.{SPINUP_START}-{SPINUP_END}.nc")

if os.path.exists(spinup_out):
    print(f"Spinup file already exists: {spinup_out}")
else:
    spinup_datasets = []
    for year in range(SPINUP_START, SPINUP_END + 1):
        mort_file = os.path.join(MORT_DIR, f"mort_{SITE}_{year}.nc")
        if not os.path.exists(mort_file):
            print(f"  WARNING: missing spinup year {year}, skipping.")
            continue
        spinup_datasets.append(xr.open_dataset(mort_file))

    if spinup_datasets:
        spinup = xr.concat(spinup_datasets, dim="time")
        spinup.to_netcdf(spinup_out)
        for ds in spinup_datasets:
            ds.close()
        print(f"Spinup file written: {spinup_out}")
    else:
        print("ERROR: No spinup datasets found. Run the processing cells first.")

Spinup file written: /home/elisacw/mimicsplus_input/Bygland/mortality/mort_Bygland_for_spinup.1850-1869.nc


## Quick sanity check

In [26]:
# Open a sample year and print variables + time range
sample_year = YEAR_START
sample_file = os.path.join(MORT_DIR, f"mort_{SITE}_{sample_year}.nc")

if os.path.exists(sample_file):
    ds_check = xr.open_dataset(sample_file)
    print(f"Variables in mort_{SITE}_{sample_year}.nc:")
    for v in ds_check.data_vars:
        print(f"  {v:35s}  shape={ds_check[v].shape}")
    ds_check.close()
else:
    print(f"Sample file not found: {sample_file}")

Variables in mort_Bygland_1850.nc:
  mcdate                               shape=(12,)
  CROOT_PROF                           shape=(12, 25, 1)
  STEM_PROF                            shape=(12, 25, 1)
  split_leaf_prof_mortC                shape=(12, 1)
  split_froot_prof_mortC               shape=(12, 1)
  split_leaf_prof_mortN                shape=(12, 1)
  split_froot_prof_mortN               shape=(12, 1)
  met_leaf_prof_mortC                  shape=(12, 1)
  met_froot_prof_mortC                 shape=(12, 1)
  met_croot_prof_mortC                 shape=(12, 1)
  met_stem_prof_mortC                  shape=(12, 1)
  met_leaf_prof_mortN                  shape=(12, 1)
  met_froot_prof_mortN                 shape=(12, 1)
  met_croot_prof_mortN                 shape=(12, 1)
  met_stem_prof_mortN                  shape=(12, 1)
